In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("oece.bronze.seace_records")
cr = "compiled_release"

# Versión más reciente de cada proceso
w = Window.partitionBy("ocid").orderBy(
    F.to_timestamp(f"{cr}.date").desc(), F.col("periodo").desc()
)
ultimos = bronze.withColumn("rn", F.row_number().over(w)).filter("rn = 1")

df_silver = ultimos.select(
    "ocid",
    F.col(f"{cr}.tender.id").alias("id_proceso"),
    F.col(f"{cr}.tender.title").alias("nomenclatura"),
    F.col(f"{cr}.tender.description").alias("descripcion"),
    F.col(f"{cr}.tender.procurementMethodDetails").alias("tipo_procedimiento"),
    F.col(f"{cr}.tender.mainProcurementCategory").alias("categoria"),
    F.to_timestamp(f"{cr}.tender.datePublished").alias("fecha_publicacion"),
    F.col(f"{cr}.tender.value.amount").cast("decimal(18,2)").alias("monto_referencial"),
    F.col(f"{cr}.tender.value.currency").alias("moneda"),
    F.col(f"{cr}.buyer.id").alias("id_entidad"),
    F.col(f"{cr}.buyer.name").alias("entidad"),
    F.expr(f"filter({cr}.parties, p -> p.id = {cr}.buyer.id)[0].address.department").alias("departamento"),
    F.size(F.coalesce(F.col(f"{cr}.tender.items"), F.array())).alias("num_items"),
    "periodo",
    "_fecha_ingesta",
)

(df_silver.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("oece.silver.procesos"))

print("bronze:", bronze.count(), "| silver:", spark.table("oece.silver.procesos").count())
display(spark.table("oece.silver.procesos").groupBy("categoria").count())

bronze: 21631 | silver: 21631


categoria,count
works,4006
goods,8714
services,8911
